# Carbon Emissions Prediction

**Tabular Regression Project** — Predict annual CO₂ emissions (tons) from industrial and energy features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `co2_tons`

## 2 · Project Overview

We predict **annual CO₂ emissions** (in metric tons) for industrial facilities based on energy usage, fuel type, workforce size, and operational characteristics. Carbon accounting and emission forecasting are critical for ESG reporting and climate policy.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given an industrial facility's energy consumption, fuel mix, workforce, floor area, machinery, waste output, vehicle fleet, renewable energy share, and operating hours, predict its **annual CO₂ emissions in tons**.

## 5 · Why This Project Matters

- **ESG compliance** requires accurate emission estimation for regulatory reporting.
- **Carbon pricing** (cap-and-trade, taxes) makes emissions a financial liability.
- Identifying emission drivers enables targeted **decarbonization strategies**.
- **Renewable energy adoption** impact can be quantified through model coefficients.
- Supports corporate **net-zero planning** with data-driven projections.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,500 |
| **Features** | 10 (energy, fuel type, employees, floor area, machinery, industry, renewable %, waste, vehicles, operating hours) |
| **Target** | `co2_tons` (continuous, metric tons CO₂/year) |
| **Categorical** | fuel_type (5), industry_sector (6) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "co2_tons"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 7500
energy_kwh = np.random.uniform(50000, 5000000, n)
fuel_type = np.random.choice(["coal", "natural_gas", "oil", "mixed", "renewable_heavy"], n,
                              p=[0.2, 0.3, 0.25, 0.15, 0.1])
employees = np.random.randint(10, 5000, n)
floor_area_m2 = np.random.uniform(200, 50000, n)
machinery_count = np.random.randint(1, 200, n)
industry = np.random.choice(["manufacturing", "chemicals", "food_processing",
                              "mining", "logistics", "tech"], n)
renewable_pct = np.random.uniform(0, 80, n)
waste_tons = np.random.uniform(10, 5000, n)
vehicles = np.random.randint(0, 300, n)
operating_hours = np.random.uniform(1000, 8760, n)

fuel_factor = np.where(fuel_type == "coal", 0.95,
              np.where(fuel_type == "oil", 0.75,
              np.where(fuel_type == "natural_gas", 0.55,
              np.where(fuel_type == "mixed", 0.65, 0.15)))).astype(float)

co2 = (energy_kwh * fuel_factor / 1000
       + employees * 2 + floor_area_m2 * 0.05
       + machinery_count * 10 + waste_tons * 0.8
       + vehicles * 15 - renewable_pct * 20
       + operating_hours * 0.3
       + np.random.normal(0, 100, n))
co2 = np.maximum(co2, 10)

df = pd.DataFrame({
    "energy_kwh": energy_kwh, "fuel_type": fuel_type,
    "employees": employees, "floor_area_m2": floor_area_m2,
    "machinery_count": machinery_count, "industry_sector": industry,
    "renewable_pct": renewable_pct, "waste_tons": waste_tons,
    "num_vehicles": vehicles, "operating_hours_yr": operating_hours,
    "co2_tons": co2
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["energy_kwh", "employees", "floor_area_m2",
                          "machinery_count", "waste_tons", "renewable_pct"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["co2_tons"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("co2_tons"); ax.set_title(f"CO₂ vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("fuel_type")["co2_tons"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Mean CO₂ by Fuel Type"); axes[0].set_xlabel("Tons CO₂")
df.groupby("industry_sector")["co2_tons"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Mean CO₂ by Industry"); axes[1].set_xlabel("Tons CO₂")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `co2_tons`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("CO₂ (tons)")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: {df[TARGET].mean():,.0f} tons, Median: {df[TARGET].median():,.0f} tons")
print(f"Std: {df[TARGET].std():,.0f} tons, Range: {df[TARGET].min():,.0f} - {df[TARGET].max():,.0f} tons")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["energy_per_employee"] = X_train["energy_kwh"] / (X_train["employees"] + 1)
X_test["energy_per_employee"] = X_test["energy_kwh"] / (X_test["employees"] + 1)

X_train["fossil_energy_proxy"] = X_train["energy_kwh"] * (1 - X_train["renewable_pct"] / 100)
X_test["fossil_energy_proxy"] = X_test["energy_kwh"] * (1 - X_test["renewable_pct"] / 100)

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Energy consumption** is the dominant predictor — more energy = more emissions.
- **Fuel type** creates large differences: coal ~6× the emissions of renewable-heavy facilities.
- **Renewable percentage** has a strong negative coefficient — each 10% increase cuts emissions significantly.
- **Vehicle fleet** and **waste** contribute meaningfully beyond stationary combustion.

**Business takeaway:** The fastest decarbonization path is switching fuel type (coal → gas → renewables). Energy efficiency reduces both cost and emissions simultaneously.

## 26 · Limitations

1. Synthetic data — real emissions depend on specific equipment efficiency and process chemistry.
2. Fuel type is simplified — real facilities use complex fuel mixes.
3. No Scope 2/3 emissions — only direct (Scope 1) is modeled.
4. No temporal trend — emission factors change as grids decarbonize.
5. No carbon capture or offset accounting.

## 27 · How to Improve This Project

1. Use real EPA Greenhouse Gas Reporting Program data.
2. Add Scope 2 (electricity grid) and Scope 3 (supply chain) emissions.
3. Include equipment age and efficiency ratings.
4. Model emission trajectories over time for net-zero planning.
5. Add carbon intensity benchmarks by industry.

## 28 · Production Considerations

- Integrate with energy management systems for automated reporting.
- Provide quarterly emission estimates for ESG disclosures.
- Monitor for anomalies indicating equipment malfunction or data errors.
- Update emission factors as grid carbon intensity changes.

## 29 · Common Mistakes

1. Not accounting for fuel type × energy interaction — same kWh from coal vs. gas yields different CO₂.
2. Treating renewable_pct as independent from energy consumption.
3. Double-counting electricity emissions (grid factor already in energy data).
4. Ignoring process emissions (e.g., cement production releases CO₂ chemically).
5. Using outdated emission factors.

## 30 · Mini Challenge / Exercises

1. Remove `fuel_type` — how much does RMSE increase?
2. Create a `carbon_intensity = co2_tons / energy_kwh` ratio — which industries are most intensive?
3. Add `energy_kwh²` — does the quadratic term improve fit?
4. Train only on coal facilities — does the model still predict gas facilities well?
5. What renewable_pct would reduce a typical facility's emissions by 50%?

## 31 · Final Summary / Key Takeaways

1. **Energy consumption × fuel type** explains most emission variation.
2. **Renewable energy share** is the strongest lever for decarbonization.
3. **Gradient boosting** captures the fuel-energy interaction better than linear models.
4. **Feature engineering** (fossil energy proxy) directly encodes the physics.
5. **Industry sector** creates meaningful baseline differences.
6. **Real deployment** needs Scope 2/3 emissions and temporal emission factor updates.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))